In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast


import os
os.chdir(r'C:\Users\AnuragS\OneDrive\cv_project\Movie Recommendation System\data')
os.getcwd()

'C:\\Users\\AnuragS\\OneDrive\\cv_project\\Movie Recommendation System\\data'

In [2]:
df = pd.read_csv('tmdb_movie_merged.csv')
df.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [8]:
print('Rows & Columns')
print(f'Rows = {df.shape[0]}')
print(f'Columns = {df.shape[1]}')
print(' ')
print('Columns are:')
print(df.columns)

Rows & Columns
Rows = 4803
Columns = 22
 
Columns are:
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'cast', 'crew'],
      dtype='str')


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [10]:
drop_cols = [
    'budget', ## not important for recommendation
    'homepage', ## 
    'original_language', ## among 5k movie 4505 are English movies (imbalanced)
    'original_title', ## language may differ, title column will be taken instead
    ## 'popularity',
    'production_companies', ## generally movies aren't recommended based on production companies
    'production_countries',
    'revenue', ## 
    'runtime',
    'spoken_languages',
    'status',
    'tagline', ## will not help
    'vote_count',
    ##'vote_average'
]

In [11]:
req_col = [col for col in df.columns if col not in drop_cols]
dfa = df[req_col].copy()

In [12]:
dfa.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   genres        4803 non-null   str    
 1   id            4803 non-null   int64  
 2   keywords      4803 non-null   str    
 3   overview      4800 non-null   str    
 4   popularity    4803 non-null   float64
 5   release_date  4802 non-null   str    
 6   title         4803 non-null   str    
 7   vote_average  4803 non-null   float64
 8   cast          4803 non-null   str    
 9   crew          4803 non-null   str    
dtypes: float64(2), int64(1), str(7)
memory usage: 375.4 KB


In [13]:
dfa.isnull().sum()

genres          0
id              0
keywords        0
overview        3
popularity      0
release_date    1
title           0
vote_average    0
cast            0
crew            0
dtype: int64

In [14]:
dfa.dropna(inplace=True)

In [15]:
dfa.duplicated().sum()

np.int64(0)

In [16]:
dfa['genres'] = dfa['genres'].apply(ast.literal_eval).apply(lambda x:[g['name']for g in x])
dfa['keywords'] = dfa['keywords'].apply(ast.literal_eval).apply(lambda x:[g['name'] for g in x])
dfa['crew'] = dfa['crew'].apply(ast.literal_eval).apply(lambda x: [g['name'] for g in x if g['job'] == 'Director'])
dfa['cast'] = dfa['cast'].apply(ast.literal_eval).apply(lambda x: [g['name'] for g in x[:3]])
dfa['overview'] = dfa['overview'].apply(lambda x: x.split())
dfa['release_date'] = pd.to_datetime(dfa['release_date'])
dfa['release_year'] = dfa['release_date'].dt.year.astype(int)

In [17]:
dfa.head(2)

,genres,id,keywords,overview,popularity,release_date,title,vote_average,cast,crew,release_year
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",150.437577,2009-12-10,Avatar,7.2,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],2009
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...","[Captain, Barbossa,, long, believed, to, be, d...",139.082615,2007-05-19,Pirates of the Caribbean: At World's End,6.9,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],2007


In [18]:
revamped_cols = ['genres','keywords','crew','cast']

for col in revamped_cols:
    dfa[col] = dfa[col].apply(lambda x:[i.replace(' ','') for i in x])

In [19]:
dfa.head(3)

,genres,id,keywords,overview,popularity,release_date,title,vote_average,cast,crew,release_year
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",150.437577,2009-12-10,Avatar,7.2,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],2009
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",139.082615,2007-05-19,Pirates of the Caribbean: At World's End,6.9,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],2007
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...","[A, cryptic, message, from, Bond’s, past, send...",107.376788,2015-10-26,Spectre,6.3,"[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],2015


In [20]:
dfa['tag'] = (dfa['overview']
             + dfa['genres']
             + dfa['overview']
             + dfa['keywords']
             + dfa['cast']
             + dfa['crew']
             #+ df['release_year']
)

In [21]:
dfa.head(2)

,genres,id,keywords,overview,popularity,release_date,title,vote_average,cast,crew,release_year,tag
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",150.437577,2009-12-10,Avatar,7.2,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],2009,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",139.082615,2007-05-19,Pirates of the Caribbean: At World's End,6.9,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],2007,"[Captain, Barbossa,, long, believed, to, be, d..."


In [22]:
df_ana = dfa[['id','title','release_year','vote_average','popularity','tag']]
df_ana

,id,title,release_year,vote_average,popularity,tag
0,19995,Avatar,2009,7.2,150.437577,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,2007,6.9,139.082615,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,2015,6.3,107.376788,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,2012,7.6,112.312950,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,2012,6.1,43.926995,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...,...,...,...
4798,9367,El Mariachi,1992,6.6,14.269792,"[El, Mariachi, just, wants, to, play, his, gui..."
4799,72766,Newlyweds,2011,5.9,0.642552,"[A, newlywed, couple's, honeymoon, is, upended..."
4800,231617,"Signed, Sealed, Delivered",2013,7.0,1.444476,"[""Signed,, Sealed,, Delivered"", introduces, a,..."
4801,126186,Shanghai Calling,2012,5.7,0.857008,"[When, ambitious, New, York, attorney, Sam, is..."


In [23]:
df_ana['tag'] = df_ana['tag'].apply(lambda x:' '.join(map(str,x))).apply(lambda y:y.lower())

In [24]:
df_ana.head()

,id,title,release_year,vote_average,popularity,tag
0,19995,Avatar,2009,7.2,150.437577,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,2007,6.9,139.082615,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,2015,6.3,107.376788,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,2012,7.6,112.312950,following the death of district attorney harve...
4,49529,John Carter,2012,6.1,43.926995,"john carter is a war-weary, former military ca..."


In [25]:
dfa.to_csv('cleaned_data.csv',index=False)
df_ana.to_csv('cleaned_data_for_anylytics.csv',index=False)

In [1]:
## the end